# 01 - EDA + CBF + CARS TourHub Bali

Notebook ini dibuat sebagai langkah awal sebelum FastAPI. Tujuannya: memahami dataset, membersihkan data, lalu mencoba rekomendasi sederhana dengan Content-Based Filtering dan Context-Aware Recommender System.


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load dataset


In [ ]:
DATA_PATH = Path('../data/bali_tourist_destination.csv')
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all'))


## 2. Cek kategori dan lokasi


In [ ]:
display(df['kategori'].value_counts())
display(df['kabupaten_kota'].value_counts())


## 3. Preprocessing sederhana


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'[^a-z0-9A-ZÀ-ÿ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def infer_tipe_wisata(row):
    name = normalize_text(row['nama_tempat_wisata'])
    kategori = normalize_text(row['kategori'])
    indoor_keywords = ['museum', 'galeri', 'gallery', 'mall', 'plaza', 'theater', 'teater', 'studio']
    outdoor_keywords = ['pantai', 'beach', 'air terjun', 'waterfall', 'bukit', 'hill', 'gunung', 'danau', 'lake', 'taman', 'park', 'sawah', 'river', 'rafting', 'snorkeling', 'diving', 'zoo', 'monkey forest', 'forest', 'pura', 'temple', 'goa', 'cave']
    if any(k in name for k in indoor_keywords):
        return 'indoor'
    if any(k in name for k in outdoor_keywords):
        return 'outdoor'
    if kategori == 'alam':
        return 'outdoor'
    if kategori == 'budaya':
        return 'mixed'
    return 'mixed'

clean = df.copy()
for col in ['id_tempat', 'nama_tempat_wisata', 'kategori', 'kecamatan', 'kabupaten_kota']:
    clean[col] = clean[col].fillna('').astype(str).str.strip()

clean['rating'] = pd.to_numeric(clean['rating'], errors='coerce').fillna(0)
clean['jumlah_rating'] = pd.to_numeric(clean['jumlah_rating'], errors='coerce').fillna(0).astype(int)
clean = clean.drop_duplicates(subset=['nama_tempat_wisata', 'latitude', 'longitude'])
clean['tipe_wisata'] = clean.apply(infer_tipe_wisata, axis=1)
clean['rating_score'] = (clean['rating'] / clean['rating'].max()).clip(0, 1)
log_reviews = np.log1p(clean['jumlah_rating'])
clean['popularity_score'] = (log_reviews / log_reviews.max()).clip(0, 1)
clean['feature_text'] = (
    clean['nama_tempat_wisata'].map(normalize_text) + ' ' +
    clean['kategori'].map(normalize_text) + ' ' +
    clean['kecamatan'].map(normalize_text) + ' ' +
    clean['kabupaten_kota'].map(normalize_text) + ' ' +
    clean['tipe_wisata'].map(normalize_text)
)
clean[['nama_tempat_wisata','kategori','kabupaten_kota','rating','jumlah_rating','tipe_wisata','feature_text']].head()


## 4. Content-Based Filtering (CBF)


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
destination_matrix = vectorizer.fit_transform(clean['feature_text'])
destination_matrix.shape


## 5. CARS: context scoring sederhana


In [ ]:
def weather_group(weather):
    text = normalize_text(weather)
    if any(x in text for x in ['hujan', 'rain', 'shower', 'storm', 'petir', 'gerimis']):
        return 'hujan'
    if any(x in text for x in ['cerah', 'clear', 'sunny']):
        return 'cerah'
    if any(x in text for x in ['berawan', 'cloud', 'mendung', 'overcast']):
        return 'berawan'
    return 'tidak_diketahui'

def context_multiplier(tipe_wisata, popularity_score, weather=None, visit_day=None, is_high_season=False):
    multiplier = 1.0
    group = weather_group(weather)
    tipe = str(tipe_wisata).lower()

    if group == 'hujan':
        if tipe == 'outdoor':
            multiplier *= 0.72
        elif tipe == 'indoor':
            multiplier *= 1.12
        else:
            multiplier *= 0.92
    elif group == 'cerah':
        if tipe == 'outdoor':
            multiplier *= 1.08
        elif tipe == 'indoor':
            multiplier *= 0.96

    if normalize_text(visit_day) in ['weekend', 'akhir pekan', 'sabtu', 'minggu']:
        multiplier *= 0.90 if popularity_score >= 0.75 else 1.03

    if is_high_season:
        multiplier *= 0.88 if popularity_score >= 0.75 else 1.05

    return multiplier


## 6. Fungsi rekomendasi final


In [ ]:
def recommend_destinations(kategori_preferensi=None, kabupaten_kota=None, kecamatan=None, keywords=None, min_rating=None, top_n=10, weather=None, visit_day=None, is_high_season=False):
    kategori_preferensi = kategori_preferensi or []
    keywords = keywords or []
    user_query = ' '.join([
        *[normalize_text(x) for x in kategori_preferensi],
        normalize_text(kabupaten_kota),
        normalize_text(kecamatan),
        *[normalize_text(x) for x in keywords],
    ]).strip() or 'wisata bali'

    user_vector = vectorizer.transform([user_query])
    cbf_scores = cosine_similarity(user_vector, destination_matrix).flatten()

    result = clean.copy()
    result['cbf_score'] = cbf_scores

    if kabupaten_kota:
        result = result[result['kabupaten_kota'].str.lower() == str(kabupaten_kota).lower()]
    if kecamatan:
        result = result[result['kecamatan'].str.lower() == str(kecamatan).lower()]
    if min_rating is not None:
        result = result[result['rating'] >= min_rating]

    result['context_multiplier'] = result.apply(
        lambda row: context_multiplier(row['tipe_wisata'], row['popularity_score'], weather, visit_day, is_high_season),
        axis=1
    )

    result['base_score'] = (0.70 * result['cbf_score']) + (0.20 * result['rating_score']) + (0.10 * result['popularity_score'])
    result['final_score'] = result['base_score'] * result['context_multiplier']

    cols = ['nama_tempat_wisata', 'kategori', 'tipe_wisata', 'kecamatan', 'kabupaten_kota', 'rating', 'jumlah_rating', 'cbf_score', 'context_multiplier', 'final_score']
    return result.sort_values('final_score', ascending=False)[cols].head(top_n)


## 7. Contoh rekomendasi


In [ ]:
recommend_destinations(
    kategori_preferensi=['Alam', 'Budaya'],
    kabupaten_kota='Kabupaten Gianyar',
    keywords=['pantai', 'sunset'],
    min_rating=4.2,
    top_n=10,
    weather='hujan',
    visit_day='weekend',
    is_high_season=False
)


## 8. Simpan dataset bersih

Dataset bersih ini bisa dipakai sebagai pembanding atau dimasukkan ke database nanti.


In [ ]:
OUTPUT_PATH = Path('../data/cleaned_bali_tourist_destination.csv')
clean.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH
